# RetentionIQ: Telecom Churn Prioritization and Revenue Protection

## Business Objective

Build a customer retention decision-support system that:

- identifies customers likely to churn,
- prioritizes high-risk and high-value customers,
- recommends targeted retention actions,
- estimates revenue at risk,
- supports retention campaign planning through an interactive dashboard.

In [ ]:
# Import the required libraries
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print("Libraries imported successfully.")

Matplotlib is building the font cache; this may take a moment.


Libraries imported successfully.


In [ ]:
# Locate the downloaded CSV
RAW_DATA_DIR = Path("../data/raw")

csv_files = list(RAW_DATA_DIR.glob("*.csv"))

print(f"CSV files found: {len(csv_files)}")

for file_path in csv_files:
    print(file_path)

CSV files found: 1
../data/raw/customer_churn_data.csv


In [ ]:
# Load the dataset
if not csv_files:
    raise FileNotFoundError(
        "No CSV file was found inside data/raw."
    )

DATA_PATH = csv_files[0]

df = pd.read_csv(DATA_PATH)

print(f"Loaded file: {DATA_PATH.name}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Loaded file: customer_churn_data.csv
Rows: 1,000
Columns: 10


In [ ]:
#display the first five records
df.head()

,CustomerID,Age,Gender,Tenure,MonthlyCharges,ContractType,InternetService,TotalCharges,TechSupport,Churn
0,1,49,Male,4,88.35,Month-to-Month,Fiber Optic,353.40,Yes,Yes
1,2,43,Male,0,36.67,Month-to-Month,Fiber Optic,0.00,Yes,Yes
2,3,51,Female,2,63.79,Month-to-Month,Fiber Optic,127.58,No,Yes
3,4,60,Female,8,102.34,One-Year,DSL,818.72,Yes,Yes
4,5,42,Male,32,69.01,Month-to-Month,NaN,"2,208.32",No,Yes


In [ ]:
#Inspect the column names
print("Column names:")

for index, column in enumerate(df.columns, start=1):
    print(f"{index}. {column}")

Column names:
1. CustomerID
2. Age
3. Gender
4. Tenure
5. MonthlyCharges
6. ContractType
7. InternetService
8. TotalCharges
9. TechSupport
10. Churn


In [7]:
#Inspect data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       1000 non-null   int64  
 1   Age              1000 non-null   int64  
 2   Gender           1000 non-null   str    
 3   Tenure           1000 non-null   int64  
 4   MonthlyCharges   1000 non-null   float64
 5   ContractType     1000 non-null   str    
 6   InternetService  703 non-null    str    
 7   TotalCharges     1000 non-null   float64
 8   TechSupport      1000 non-null   str    
 9   Churn            1000 non-null   str    
dtypes: float64(2), int64(3), str(5)
memory usage: 78.3 KB


In [8]:
df.dtypes

CustomerID           int64
Age                  int64
Gender                 str
Tenure               int64
MonthlyCharges     float64
ContractType           str
InternetService        str
TotalCharges       float64
TechSupport            str
Churn                  str
dtype: object

In [9]:
#Check missing values
missing_summary = (
    df.isna()
    .sum()
    .to_frame(name="missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df) * 100
)

missing_summary

,missing_count,missing_percentage
CustomerID,0,0.00
Age,0,0.00
Gender,0,0.00
Tenure,0,0.00
MonthlyCharges,0,0.00
ContractType,0,0.00
InternetService,297,29.70
TotalCharges,0,0.00
TechSupport,0,0.00
Churn,0,0.00


In [10]:
# Check for empty strings
empty_string_counts = {}

for column in df.select_dtypes(include="object").columns:
    empty_string_counts[column] = (
        df[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

pd.Series(
    empty_string_counts,
    name="empty_string_count",
).sort_values(ascending=False)

/var/folders/nq/gr9z11zn4c721wlc5y_rb0y80000gn/T/ipykernel_28731/1642807793.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df.select_dtypes(include="object").columns:


Gender             0
ContractType       0
InternetService    0
TechSupport        0
Churn              0
Name: empty_string_count, dtype: int64

Check duplicate rows and customer IDs

In [ ]:
print(f"Fully duplicated rows: {df.duplicated().sum()}")

Fully duplicated rows: 0


In [12]:
customer_id_columns = [
    column
    for column in df.columns
    if "customer" in column.lower() and "id" in column.lower()
]

print("Possible customer ID columns:", customer_id_columns)

Possible customer ID columns: ['CustomerID']


In [13]:
if customer_id_columns:
    customer_id_column = customer_id_columns[0]

    print(
        "Duplicated customer IDs:",
        df[customer_id_column].duplicated().sum(),
    )

    print(
        "Unique customer IDs:",
        df[customer_id_column].nunique(),
    )

Duplicated customer IDs: 0
Unique customer IDs: 1000


Inspect categorical values

In [14]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print(f"Categorical columns: {len(categorical_columns)}")
print(categorical_columns)

Categorical columns: 5
['Gender', 'ContractType', 'InternetService', 'TechSupport', 'Churn']


/var/folders/nq/gr9z11zn4c721wlc5y_rb0y80000gn/T/ipykernel_28731/27800939.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns.tolist()


In [15]:
for column in categorical_columns:
    print(f"\n{'=' * 60}")
    print(f"Column: {column}")
    print(df[column].value_counts(dropna=False))


Column: Gender
Gender
Female    538
Male      462
Name: count, dtype: int64

Column: ContractType
ContractType
Month-to-Month    511
One-Year          289
Two-Year          200
Name: count, dtype: int64

Column: InternetService
InternetService
Fiber Optic    395
DSL            308
NaN            297
Name: count, dtype: int64

Column: TechSupport
TechSupport
Yes    506
No     494
Name: count, dtype: int64

Column: Churn
Churn
Yes    883
No     117
Name: count, dtype: int64


Inspect numerical columns

In [16]:
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()

print(f"Numerical columns: {len(numerical_columns)}")
print(numerical_columns)

Numerical columns: 5
['CustomerID', 'Age', 'Tenure', 'MonthlyCharges', 'TotalCharges']


In [17]:
df[numerical_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
CustomerID,"1,000.00",500.50,288.82,1.00,250.75,500.50,750.25,"1,000.00"
Age,"1,000.00",44.67,9.80,12.00,38.00,45.00,51.00,83.00
Tenure,"1,000.00",18.97,18.89,0.00,5.00,13.00,26.00,122.00
MonthlyCharges,"1,000.00",74.39,25.71,30.00,52.36,74.06,96.10,119.96
TotalCharges,"1,000.00","1,404.36","1,571.76",0.00,345.22,872.87,"1,900.18","12,416.25"


Check for invalid numerical values

In [18]:
for column in numerical_columns:
    print(
        f"{column}: "
        f"minimum={df[column].min()}, "
        f"maximum={df[column].max()}"
    )

CustomerID: minimum=1, maximum=1000
Age: minimum=12, maximum=83
Tenure: minimum=0, maximum=122
MonthlyCharges: minimum=30.0, maximum=119.96
TotalCharges: minimum=0.0, maximum=12416.25


Inspect the churn target

In [19]:
churn_columns = [
    column
    for column in df.columns
    if "churn" in column.lower()
]

print("Possible churn columns:", churn_columns)

Possible churn columns: ['Churn']


In [20]:
if churn_columns:
    churn_column = churn_columns[0]

    churn_counts = df[churn_column].value_counts(dropna=False)
    churn_percentages = (
        df[churn_column]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

    churn_summary = pd.DataFrame({
        "customer_count": churn_counts,
        "percentage": churn_percentages,
    })

    churn_summary

Create the first data-quality summary

In [21]:
quality_summary = pd.DataFrame({
    "metric": [
        "Rows",
        "Columns",
        "Missing values",
        "Duplicate rows",
        "Unique customer IDs",
    ],
    "value": [
        df.shape[0],
        df.shape[1],
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        (
            df[customer_id_columns[0]].nunique()
            if customer_id_columns
            else "Not identified"
        ),
    ],
})

quality_summary

,metric,value
0,Rows,1000
1,Columns,10
2,Missing values,297
3,Duplicate rows,0
4,Unique customer IDs,1000


In [22]:
print("=" * 70)
print("1. DATASET SHAPE")
print("=" * 70)
print(df.shape)

print("\n" + "=" * 70)
print("2. COLUMN NAMES")
print("=" * 70)
print(df.columns.tolist())

print("\n" + "=" * 70)
print("3. DATAFRAME INFO")
print("=" * 70)
df.info()

print("\n" + "=" * 70)
print("4. MISSING-VALUE SUMMARY")
print("=" * 70)
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})
print(missing_summary)

print("\n" + "=" * 70)
print("5. DUPLICATES")
print("=" * 70)
print("Duplicate rows:", df.duplicated().sum())

customer_id_columns = [
    column
    for column in df.columns
    if "customer" in column.lower() and "id" in column.lower()
]

if customer_id_columns:
    customer_id_column = customer_id_columns[0]
    print("Customer ID column:", customer_id_column)
    print(
        "Duplicated customer IDs:",
        df[customer_id_column].duplicated().sum()
    )
else:
    print("Customer ID column not automatically identified.")

print("\n" + "=" * 70)
print("6. CHURN COUNTS AND PERCENTAGES")
print("=" * 70)

churn_columns = [
    column
    for column in df.columns
    if "churn" in column.lower()
]

if churn_columns:
    churn_column = churn_columns[0]

    churn_summary = pd.DataFrame({
        "count": df[churn_column].value_counts(dropna=False),
        "percentage": (
            df[churn_column]
            .value_counts(normalize=True, dropna=False)
            .mul(100)
            .round(2)
        )
    })

    print("Churn column:", churn_column)
    print(churn_summary)
else:
    print("Churn column not automatically identified.")

1. DATASET SHAPE
(1000, 10)

2. COLUMN NAMES
['CustomerID', 'Age', 'Gender', 'Tenure', 'MonthlyCharges', 'ContractType', 'InternetService', 'TotalCharges', 'TechSupport', 'Churn']

3. DATAFRAME INFO
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       1000 non-null   int64  
 1   Age              1000 non-null   int64  
 2   Gender           1000 non-null   str    
 3   Tenure           1000 non-null   int64  
 4   MonthlyCharges   1000 non-null   float64
 5   ContractType     1000 non-null   str    
 6   InternetService  703 non-null    str    
 7   TotalCharges     1000 non-null   float64
 8   TechSupport      1000 non-null   str    
 9   Churn            1000 non-null   str    
dtypes: float64(2), int64(3), str(5)
memory usage: 78.3 KB

4. MISSING-VALUE SUMMARY
                 missing_count  missing_percentage
CustomerID        